<a href="https://colab.research.google.com/github/yenlung/Python-Math-AI/blob/main/14%E9%81%B7%E7%A7%BB%E5%BC%8F%E5%AD%B8%E7%BF%92%E5%81%9A%E5%85%AB%E5%93%A5%E8%BE%A8%E8%AD%98%E5%99%A8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 14 遷移式學習做八哥辨識器 🦜
## CNN × 遷移學習 × 30 張照片打造圖片分類 App

上週的全連結網路訓練了 MNIST，但如果你拿著這個模型去辨識「貓vs狗」，完全沒用——因為它只認識 784 個像素，不懂「特徵」。

真正辨識圖片的是 **CNN（卷積神經網路）**。

但訓練一個好的 CNN，通常需要：
- 📁 幾萬張以上的訓練圖片
- ⏱️ 幾天的訓練時間
- 💰 高貴的 GPU 費用

---

## 🪄 遷移學習的魔法

> 「**站在巨人的肩膀上。**」

我們不從頭訓練，而是**借用**已經訓練好的神經網路！

**ResNet50V2** 在 1,400 萬張圖片上訓練了 1,000 個類別的辨識，它已經懂得：
- 邊緣、紋理、形狀、顏色
- 眼睛、羽毛、翅膀的特徵
- ...幾乎所有視覺特徵

我們只需要把最後一層「換成我們的分類問題」，讓它用 30 張照片學習「哪種特徵對應哪種八哥」！

---

### 今天的目標：
- 🦅 認識三種台灣常見八哥：土八哥、白尾八哥、家八哥
- 🧠 用 **30 張以內**的照片，打造準確率接近 100% 的辨識器
- 🌐 打造可以分享的 **Gradio 圖片辨識 Web App**

---

### ✨ 更厲害的是：這個框架可以辨識任何你想辨識的東西！

只要換一組照片，你可以做：
- 🌸 花卉辨識器
- 🐱 貓品種辨識器
- 🍱 台灣小吃辨識器
- 👗 服飾風格分類器

## 🤖 AI 可以怎麼幫你？

你可以問 AI：

- CNN（卷積神經網路）是什麼？
- 遷移學習是什麼？為什麼有效？
- ResNet 為什麼叫「殘差網路」？
- `include_top=False` 是什麼意思？
- 為什麼 30 張照片就夠？
- 我想做自己的分類器，要怎麼準備資料？

但記得：**先自己想，再問 AI**

## 0. 個人化設定（你可以直接改這裡！）

改成你自己想辨識的類別，這個 App 就完全屬於你了！

In [ ]:
# 辨識類別（英文、無空格，用逗號分隔）
# 每個類別建立同名資料夾，放入照片
category_en = "crested_myna,javan_myna,common_myna"

# 類別的中文名稱（顯示用）
category_zh = "土八哥,白尾八哥,家八哥"

# App 標題與說明
app_title = "🦜 台灣八哥辨識器"
app_description = "請上傳一張八哥照片，AI 會告訴你是哪種八哥！"

## 1. 安裝與讀入套件

In [ ]:
!pip install -q gradio

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import os
import zipfile
import PIL.Image as Image
import gradio as gr

from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras.applications.resnet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical

print('套件讀入完成！')

In [ ]:
categories = category_en.split(',')
labels = category_zh.split(',')
N = len(categories)
base_dir = '/content/'

print(f'分類數量: {N}')
print(f'類別: {labels}')

## 2. 準備訓練照片

### 方法 A：使用我們預備的八哥照片

直接下載預備好的三種八哥照片（各約 9~16 張）。

### 方法 B：自己準備照片

在 Colab 左側建立和 `category_en` 同名的資料夾，把照片放進去。

**照片準備小技巧：**
- 每個類別 **5~20 張**就夠（遷移學習的威力！）
- 照片越多樣越好（不同角度、光線、背景）
- 可以直接 Google 圖片搜尋然後儲存

In [ ]:
# 建立各類別資料夾
for category in categories:
    os.makedirs(f'./{category}', exist_ok=True)
    print(f'建立資料夾: {category}')

In [ ]:
# 下載八哥照片（如果自己準備照片，跳過這格）
!wget -q --no-check-certificate \
    https://github.com/yenlung/AI-Demo/raw/master/myna.zip \
    -O /content/myna.zip

local_zip = '/content/myna.zip'
zip_ref = zipfile.ZipFile(local_zip, 'r')
zip_ref.extractall('/content')
zip_ref.close()

print('照片下載並解壓完成！')

# 列出各類別有多少張照片
for i, cat in enumerate(categories):
    n_photos = len(os.listdir(base_dir + cat))
    print(f'{labels[i]}（{cat}）: {n_photos} 張')

### 預覽訓練照片

In [ ]:
fig, axes = plt.subplots(N, 5, figsize=(15, 4*N))

for i, (cat, label) in enumerate(zip(categories, labels)):
    thedir = base_dir + cat
    fnames = os.listdir(thedir)[:5]  # 只顯示前 5 張

    for j, fname in enumerate(fnames):
        img = Image.open(f'{thedir}/{fname}')
        axes[i][j].imshow(img)
        axes[i][j].set_title(label if j == 0 else '', fontsize=12)
        axes[i][j].axis('off')

plt.suptitle('訓練照片預覽', fontsize=15)
plt.tight_layout()
plt.show()

## 3. 準備訓練數據

把所有照片讀進來，統一縮放成 224×224（ResNet 的輸入大小）：

In [ ]:
data = []
target = []

for i, cat in enumerate(categories):
    thedir = base_dir + cat
    file_names = os.listdir(thedir)
    for fname in file_names:
        img_path = f'{thedir}/{fname}'
        img = load_img(img_path, target_size=(224, 224))
        x = img_to_array(img)
        data.append(x)
        target.append(i)

data = np.array(data)
print(f'總共 {len(data)} 張照片，每張 {data.shape[1]}×{data.shape[2]}，{data.shape[3]} 個顏色通道')

In [ ]:
# ResNet 有自己的前處理方式（不是簡單的 /255）
x_train = preprocess_input(data)

# One-Hot Encoding
y_train = to_categorical(target, N)

print('x_train shape:', x_train.shape)
print('y_train shape:', y_train.shape)
print('y_train 範例:', y_train[0])

## 4. 打造遷移學習神經網路

### 理解 ResNet50V2

ResNet = **Res**idual **Net**work（殘差網路）

- 2015 年 ImageNet 競賽冠軍
- 50 層卷積神經網路
- 在 1,400 萬張圖片上訓練，學會識別 1,000 種物體
- 已有 **2,500 萬個參數**幫你抽取圖片特徵

我們要做的事：
1. 去掉 ResNet 最後的輸出層（原本輸出 1,000 類）
2. 接上我們自己的輸出層（輸出 N 類）
3. **只訓練我們自己的輸出層**（6,000 個參數，而不是 2,500 萬個）

In [ ]:
# 載入 ResNet50V2（去掉最後的全連結層，加上全域平均池化）
resnet = ResNet50V2(include_top=False, pooling="avg")

# 凍結 ResNet 的所有層（不重新訓練）
resnet.trainable = False

print(f'ResNet50V2 載入完成！')
print(f'ResNet 輸出維度: {resnet.output_shape}')  # (None, 2048)

In [ ]:
# 建立我們的遷移學習模型
model = Sequential([
    resnet,                          # 借用 ResNet 的特徵提取能力
    Dense(N, activation='softmax')   # 我們自己的分類層
])

model.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

model.summary()

### 數字的意義

- **Non-trainable params**：ResNet 的 2,300 萬個參數，我們借用但不動它
- **Trainable params**：只有我們自己的輸出層，約 6,000 個參數！

這就是為什麼 **30 張照片就夠**！我們只需要學最後一小步。

## 5. 訓練！

因為參數很少，訓練非常快（幾秒到幾分鐘）：

In [ ]:
history = model.fit(x_train, y_train,
                    batch_size=10,
                    epochs=20,
                    verbose=1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], 'o-', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('訓練正確率')
axes[0].set_ylim(0, 1.05)
axes[0].grid(True)

axes[1].plot(history.history['loss'], 's-', color='orange', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('訓練 Loss')
axes[1].grid(True)

plt.suptitle('遷移學習訓練過程', fontsize=13)
plt.tight_layout()
plt.show()

## 6. 評估：看看學得怎麼樣

In [ ]:
loss, acc = model.evaluate(x_train, y_train, verbose=0)
print(f'訓練集 Loss:    {loss:.4f}')
print(f'訓練集正確率: {acc*100:.2f}%')

In [ ]:
# 查看每張照片的預測結果
y_pred = np.argmax(model.predict(x_train, verbose=0), axis=-1)
y_true = np.array(target)

n_correct = np.sum(y_pred == y_true)
print(f'{len(y_true)} 張照片中，正確辨識了 {n_correct} 張')

In [ ]:
# 顯示部分預測結果
n_show = min(len(data), 12)
indices = np.random.choice(len(data), n_show, replace=False)

fig, axes = plt.subplots(2, 6, figsize=(18, 6))

for plot_i, data_i in enumerate(indices):
    ax = axes[plot_i//6][plot_i%6]
    ax.imshow(data[data_i].astype(np.uint8))
    pred_label = labels[y_pred[data_i]]
    true_label = labels[y_true[data_i]]
    color = 'green' if y_pred[data_i] == y_true[data_i] else 'red'
    ax.set_title(f'預測: {pred_label}\n真實: {true_label}',
                fontsize=9, color=color)
    ax.axis('off')

plt.suptitle('辨識結果（綠=正確，紅=錯誤）', fontsize=13)
plt.tight_layout()
plt.show()

## 7. 打造 Gradio 圖片辨識 Web App！🎉

用這個 App，任何人都能上傳照片讓 AI 辨識！

In [ ]:
def resize_image(inp):
    """把輸入圖片調整成 224×224 的 numpy array"""
    img = Image.fromarray(inp)
    img_resized = img.resize((224, 224), Image.Resampling.LANCZOS)
    return np.array(img_resized)


def classify_image(inp):
    """輸入圖片，輸出各類別機率"""
    img_array = resize_image(inp)
    inp_processed = preprocess_input(img_array.reshape(1, 224, 224, 3))
    prediction = model.predict(inp_processed, verbose=0).flatten()
    return {labels[i]: float(prediction[i]) for i in range(N)}

In [ ]:
# 從訓練資料中選幾張照片當範例
sample_images = []
for cat in categories:
    thedir = base_dir + cat
    for fname in os.listdir(thedir)[:2]:  # 每類取 2 張
        sample_images.append(f'{thedir}/{fname}')

print(f'已準備 {len(sample_images)} 張範例圖片')

In [ ]:
with gr.Blocks(title=app_title, theme=gr.themes.Soft()) as demo:
    gr.Markdown(f"# {app_title}")
    gr.Markdown(app_description)

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(label="請上傳一張照片")
        with gr.Column():
            label_output = gr.Label(
                num_top_classes=N,
                label="AI 辨識結果"
            )

    classify_btn = gr.Button("🔍 開始辨識", variant="primary")
    classify_btn.click(fn=classify_image,
                       inputs=image_input,
                       outputs=label_output)

    gr.Examples(
        examples=sample_images,
        inputs=image_input,
        label="點擊試試範例照片"
    )

    gr.Markdown("---")
    gr.Markdown(
        f"*類別：{'、'.join(labels)}　|　模型：ResNet50V2 遷移學習*"
    )

demo.launch(share=True, debug=True)

## 💡 進階：Fine-tuning（微調）

如果你的類別和 ResNet 訓練的差異很大（例如辨識 X 光片、顯微鏡圖），
可以試試**解凍部分 ResNet 層**，讓它也學習調整。

注意：需要更多照片，訓練也更慢！

In [ ]:
# Fine-tuning 範例（先確保基本訓練完成後再執行）

# 解凍最後幾層
# resnet.trainable = True
# for layer in resnet.layers[:-20]:
#     layer.trainable = False

# 用更小的 learning rate 重新訓練
# model.compile(loss='categorical_crossentropy',
#               optimizer=tf.keras.optimizers.Adam(1e-5),
#               metrics=['accuracy'])

# model.fit(x_train, y_train, epochs=10, batch_size=10)

print('Fine-tuning 需要更多照片，謹慎使用！')
print('建議先用基本的遷移學習確認模型能正常學習後再嘗試。')

# 🧠 核心觀念回顧

```
遷移學習的概念

ResNet50V2（已訓練好）
    ↓
特徵提取器（凍結，不訓練）
    ↓
我們的分類器（只訓練這層！）
    ↓
N 個類別的機率
```

| 比較 | 從頭訓練 | 遷移學習 |
|------|----------|----------|
| 所需照片 | 數萬張 | **5~20 張** |
| 訓練時間 | 數小時 | **幾分鐘** |
| 準確率 | 高（如果資料夠多）| 高（借用大量知識）|
| 適合場景 | 有大量標記資料 | **小資料、快速原型** |

# 🎯 本週創作任務

### 🌟 打造你自己的圖片辨識器！

選一個你有興趣的主題，收集照片，用遷移學習打造專屬 App：

- 🌸 **花卉辨識**：玫瑰 vs 向日葵 vs 鬱金香
- 🍱 **台灣美食**：滷肉飯 vs 蚵仔煎 vs 珍珠奶茶
- 🐱 **貓狗品種**：柴犬 vs 柯基 vs 哈士奇
- 🎭 **名人辨識**：你喜歡的偶像（只要有照片就行！）
- 🃏 **自選任意分類**

**步驟：**
1. 修改 cell 0 的設定（類別名稱、App 標題）
2. 收集每類 5~15 張照片放入對應資料夾
3. 執行全部 cell
4. 分享你的 Gradio App 連結！

回答：

1️⃣ 你辨識的是什麼？每類幾張照片？
2️⃣ 訓練後的正確率是多少？
3️⃣ 有沒有什麼照片被辨識錯？為什麼？
4️⃣ AI 如何幫助你完成？